This notebook organizes and formats results for downstream plotting functions. 

Specifically, it generates plots for

```
- Fig 2B
- Fig 3C
- Fig 4
- Supp Fig S4
- Supp Fig S5
- Supp Figs S12
```



## 0. Mount Drive and set the main path

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries

In [ ]:
import os
import re
import pickle
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, drive_dir)

import plot_results, kMers

## 2. Get results and create a base dataframe for further manipulations

In [ ]:
# load results for each iteration
ll = {f'iter_{i}': pd.read_csv(f'{drive_dir}_saved/iteration_{i}/results/non_deep/iter_{i}.csv', index_col=0) for i in range(1,6)}
for key in ll.keys():
  ll[key] = ll[key].rename(columns = {'seed' : 'series'}) #necessary to use all downstream functions

# compute mean across iterations
ml_res = ll['iter_1'].copy()
for key in list(ll.keys())[1:]:
  ml_res = ml_res.add(ll[key])

ml_res = ml_res.apply(lambda x : x/5)
ml_res['series'] = ml_res.series.apply(lambda x : int(x))

### 2.1 Extended heatmap for Fig 2B

Plots accuracy (R2) for combinations of train sizes, encodings, and non-deep regressors for all mutational series.

In [ ]:
plt.subplots(figsize=(18,8))
sns.heatmap(ml_res.drop(columns=['series']),
            cmap='coolwarm',
            vmin=0, vmax=1,
            cbar=True, 
            )

### 2.2 Plot one panel from figure 2B

```
- sr   (->int): mutational series index; choose from the range [1,56]
- splt (->float): percentage of total sequences for training; choose one from {0.05, 0.1, 0.25, 0.5, 0.75}
```

In [ ]:
sr, spl = 4, 0.75
plot_results.library_heatmap(results=ml_res,
                             series=sr,
                             split=spl,
                             annot=True,
                             xtick=False,
                             ytick=False,
                             cbar_bool=False
                             )

### 2.3 Plot cross-validation of non-deep models in Supplementary Figure S4.

In [ ]:
sr = 4 # mutational series index; choose from the range [1,56]

iter_1_lst, iter_2_lst, iter_3_lst, iter_4_lst, iter_5_lst,  = [], [], [], [], []
for i in range(20):
  iter_1_lst.extend(ll['iter_1'].loc[ll['iter_1'].series == float(sr)].drop(columns=['series']).T.iloc[i])
  iter_2_lst.extend(ll['iter_2'].loc[ll['iter_2'].series == float(sr)].drop(columns=['series']).T.iloc[i])
  iter_3_lst.extend(ll['iter_3'].loc[ll['iter_3'].series == float(sr)].drop(columns=['series']).T.iloc[i])
  iter_4_lst.extend(ll['iter_4'].loc[ll['iter_4'].series == float(sr)].drop(columns=['series']).T.iloc[i])
  iter_5_lst.extend(ll['iter_5'].loc[ll['iter_5'].series == float(sr)].drop(columns=['series']).T.iloc[i])

hue_lst, mean_plt_lst = [], []
for j in range(20):
  hue_lst.extend(['one-hot', '1-mers', '4-mers', '4-counts', 'features', 'mixed'])
  mean_plt_lst.extend(ml_res.loc[(ml_res.series == sr)].drop(columns=['series']).T.iloc[j].values.tolist())

plt.subplots(figsize = (13,6))
for j in [iter_1_lst, iter_2_lst, iter_3_lst, iter_4_lst, iter_5_lst]:
  sns.scatterplot(x = [i for i in range(120)],
                  y = j,
                  size = .5,
                  #linewidth = 0,
                  hue = hue_lst)
  plt.ylim((0, 1))
  plt.xlim((-1, 120))
  plt.legend([])
sns.scatterplot([i for i in range(120)], 
                mean_plt_lst, 
                marker = '_',
                palette = 'gray',
                linewidth = 3,
                size = 3)
plt.legend([])
area_lst = [q for q in range(0,120,6)]
for q in range(0, len(area_lst), 2):
  plt.fill_between([area_lst[q], area_lst[q+1]], 
                   [1, 1], 
                   alpha = 0.1)

### 2.4 Remove all splits for ridge regression and low-N splits to generate swarmplots (Fig S5)

In [ ]:
drop_list = [f'ridge_{sz}' for sz in [0.05, 0.1, 0.25, 0.5, 0.75]]
drop_list.append('series')
nn = ml_res.drop(columns=drop_list)

swarms_dict = plot_results.swarms_format(nn, split_lst=['5', '10', '25', '50', '75'])

ordered_swarms_dict = {}
for key in swarms_dict.keys():
  ordered_swarms_dict[key] = pd.DataFrame()
  for o in range(0, 1008, 6):
    tt = pd.DataFrame({'R2' : [swarms_dict[key].iloc[o+4].R2,
                               swarms_dict[key].iloc[o+2].R2,
                               swarms_dict[key].iloc[o+1].R2,
                               swarms_dict[key].iloc[o+5].R2,
                               swarms_dict[key].iloc[o+3].R2,
                               swarms_dict[key].iloc[o].R2
                               ],
                      'encoding'  : ['features', '4-mers', '1-mers', 'mixed', '4-counts', 'one-hot'],
                      'regressor' : swarms_dict[key].regressor[o:o+6].to_list() 
                        })
    ordered_swarms_dict[key] = pd.concat((ordered_swarms_dict[key], tt), axis=0)

for key in list(ordered_swarms_dict.keys()):
  plot_results.swarmbox_plots(ordered_swarms_dict[key], 
                              fig_sz=(9,4), 
                              points_sz=2.4, 
                              legend=False)
  plt.ylim((-0.6, 1.0))

## 3. CNNs

### 3.1 Plot comparison of non-deep and deep models in Figure 3C.

In [ ]:
# load results for each iteration
pp = {f'iter_{i}': pd.read_csv(f'{drive_dir}_saved/iteration_{i}/results/deep/cnn_res_iter_{i}.csv', index_col=0) for i in range(1,6)}

# compute mean across iterations
cnn_res = pp['iter_1'].copy()
for key in list(pp.keys())[1:]:
  cnn_res = cnn_res.add(pp[key])

cnn_res = cnn_res.apply(lambda x : x/5)
cnn_res['series'] = cnn_res.series.apply(lambda x : int(x))

##
# drop regressors, encodings, and train sizes we don't need for Fig 3C
nn = nn.drop(columns=[f'SVR_{sz}' for sz in [0.05, 0.1, 0.25, 0.5, 0.75]])
nn = nn.drop(columns=list(itertools.chain(*[[f'{reg}_{sz}' for reg in ['MLP', 'RF']] for sz in [0.05, 0.1]])))
qq = nn.T.drop(columns=['1-mers', '4-mers', '4-counts', 'features', 'mixed'])
qq = pd.concat((qq, cnn_res.drop(columns='series').T))

viol_df = pd.DataFrame()
for col in qq.T.columns:
  temp_df = pd.DataFrame({'R2' : qq.T[col],
                          'encoding'  : ['one-hot' for itr in range(56)],
                          'regressor' : [col[:2] if col[:2]=='RF' else col[:3] for itr in range(56)],
                          'tr_size'   : [float(col[3:]) if col[:3]=='RF_' else float(col[4:]) for itr in range(56)]})
  viol_df = pd.concat((viol_df, temp_df), axis=0).reset_index(drop=True)

plt.subplots(figsize=(10,5))
sns.violinplot(x=viol_df.tr_size, 
               y='R2',
               hue = 'regressor', 
               palette="Set2",
               data=viol_df,
               linewidth=1
               )
plt.ylim((0,1))

## 4. Sequence diversity

In [ ]:
# This block produces a dataframe that is used to generate Figs 4 and S12.

itr_id = 1 #choose a value in [1,5] to load different iterations of the diversity scheme
LOAD_DIR = f'{drive_dir}_saved/iteration_{itr_id}/results/diversity/'

# load data
cnn_oh_dbt = pd.read_csv(f'{LOAD_DIR}DbT_cnn_oh_rpt{itr_id}.csv', index_col=0)
cnn_oh_dbt.index = [itr for itr in range(1, 57)]

with open(f'{LOAD_DIR}div_itr_{itr_id}.npy', 'rb') as f:
    idx_arr = np.load(f) 

# get order of mutational series
idx_lst = list(idx_arr)
ll = [itr for itr in range(1,57)]
for elm in ll:
  if elm not in idx_lst:
    idx_lst.extend([elm])

# dataframe with results from 27 models
cnn_oh_dbt = cnn_oh_dbt.loc[idx_lst]

### 4.1 Heatmap of in- and cross-series accuracies for 27 models (Figs 4A & S12; left)

In [ ]:
plt.subplots(figsize = (13,5))
sns.heatmap(cnn_oh_dbt.T,
            cmap=sns.cubehelix_palette(start=2, rot=0, dark=0.05, light=.98, reverse=False, as_cmap=True),
            center=0.5,
            vmin=0,
            vmax=1)

### 4.2 Prediction accuracy across mutational series included in the training data (Fig not included in the paper).

In [ ]:
nn, idx = [], 1
for col in cnn_oh_dbt.columns:
  temp_av = np.mean(cnn_oh_dbt[col].to_list()[:idx*2])
  nn.append(temp_av)
  idx+=1
  
plt.bar([itr for itr in range(1,28)], nn)
plt.ylabel('R2')
plt.xlabel('model')
plt.ylim((0.0, 1))

### 4.3 Diversity of training sequences
Counts the occurance of unique overlapping 5-mers across all sequences of each training set, and quantified as $1/\sum_{i=1}^{100}c_i$, where $c_i$ is the count of the top $i$ 5-mers. Specifically: 

```
- change k (->int) to test different k-mer lengths; default is 5-mers
- change *top_N* of top unique *k*-mers to generate a barplot sorted by increasing diversity (not included in the paper)

```

In [ ]:
PATH_DIV = f'{drive_dir}_saved/iteration_{itr_id}/results/diversity/exp_dict.pkl' #create path string

# load pickle object into dictionary
file = open(f"{PATH_DIV}",'rb')
exp_dict = pickle.load(file)

def rename_df_col(df, old, new):
  return df.rename(columns={old : new})

# rename to improve handling
exp_freq_dict, kmer = dict(), 5
for exp in exp_dict.keys():
  exp_dict[exp]['train']['seed'] = [1 for itr in range(exp_dict[exp]['train'].shape[0])]
  exp_dict[exp]['train'] = rename_df_col(exp_dict[exp]['train'], 'sequence', 'Sequence')
  exp_freq_dict[f'kFreq_{exp}'] = kMers.kmer_profiles(1, {'seed_1' : exp_dict[exp]['train']}, kmer)

# create pandas dataframe for downstream processing and plotting
nnn = pd.DataFrame({'kmers'  : exp_freq_dict['kFreq_exp_1'].keys(),
                    'ID'     : [id for id in range(4**kmer)],
                    'counts' : [vl for vl in exp_freq_dict['kFreq_exp_1'].values()],
                    'exp'    : ['0' for exp_n in range(4**kmer)]
                    })

for itr, dat in enumerate(list(exp_freq_dict.keys())[1:]):
  nnn = pd.concat((nnn, pd.DataFrame({'kmers'  : exp_freq_dict['kFreq_exp_1'].keys(),
                                      'ID'     : [id for id in range(4**kmer)],
                                      'counts' : [vl for vl in exp_freq_dict[dat].values()],
                                      'exp'    : [f'{itr+1}' for exp_n in range(4**kmer)]})), axis=0).reset_index(drop=True)

new, new_sort = pd.DataFrame(), pd.DataFrame()
for itr in range(len(exp_dict)):
  new[f'exp_{itr}'] = nnn[nnn.exp == f'{itr}'].counts.reset_index(drop=True)

sort_based = 'all'
if sort_based == 'exp_0':
  new_sort = new.sort_values(by=sort_based,ascending=False).reset_index(drop=True)
else:
  for col in new:
    new_sort[col] = new[col].sort_values(ascending=False, ignore_index=True) 

# plot
top_N = 100 #default value of top 100 5-mers, as shown in Fig 5
plt.bar([itr for itr in range(1, 28)], new_sort.loc[:top_N].sum().apply(lambda x : 1/x).values)
plt.ylabel('diversity')
plt.xlabel('model')

Compute mean R2 across *in-* and *cross-*series heldout test sets, and generate bubble plot (Figs 4B & S12, right)

In [ ]:
cr, idx = [], 1
for col in cnn_oh_dbt.columns:
  temp_av = np.mean(cnn_oh_dbt[col].to_list())
  cr.append(temp_av)
  idx+=1
  
nn = pd.DataFrame({'diversity' : (new_sort.loc[:top_N].sum().apply(lambda x : 1/x).values),
                   'sequences/series' : [int(5800/no_sr) for no_sr in range(2, 56, 2)],
                   'accuracy' : cr})
nn['accuracy'] = nn.accuracy.apply(lambda x: 0 if x<0 else x)


sns.relplot(x="diversity", 
            y="sequences/series", 
            hue="accuracy", 
            size="accuracy",
            sizes=(20, 200), 
            alpha=.85, 
            palette="coolwarm",
            height = 5, 
            aspect = 1.5,
            data=nn)

plt.ylim((0, 3000))
plt.xlim((4e-6, 10e-6))